# Lesson 6.1 - MLOps Fundamentals & Lifecycle (conceptual demo)

This notebook demonstrates a compact, end-to-end ML lifecycle loop: data load, train/validate, experiment logging, and artifact packaging.

## Objectives

- Build a minimal but reproducible training workflow.
- Log metrics and parameters with MLflow when available.
- Persist model artifacts with version-like metadata.
- Connect notebook steps to CI/CD/CT/CM concepts.

## Setup & Reproducibility

In [1]:
from __future__ import annotations

import json
import random
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ARTIFACT_DIR = Path("artifacts/lesson6_1")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## Section A - Data Ingestion and Versioned Snapshot

In [2]:
raw = load_breast_cancer(as_frame=True)
df = raw.frame.copy()

snapshot_path = ARTIFACT_DIR / "data_snapshot.csv"
df.to_csv(snapshot_path, index=False)

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

{"rows": len(df), "features": X.shape[1], "snapshot": str(snapshot_path)}

{'rows': 569,
 'features': 30,
 'snapshot': 'artifacts/lesson6_1/data_snapshot.csv'}

## Section B - Training + Evaluation (Experiment Unit)

In [3]:
pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=300, random_state=SEED)),
    ]
)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "f1": float(f1_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_prob)),
}
metrics

{'accuracy': 0.9824561403508771,
 'f1': 0.9861111111111112,
 'roc_auc': 0.9953703703703703}

## Section C - MLflow Logging (Optional) + Artifact Packaging

In [4]:
run_metadata = {
    "run_timestamp": datetime.utcnow().isoformat() + "Z",
    "seed": SEED,
    "model": "logistic_regression",
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "metrics": metrics,
}

model_path = ARTIFACT_DIR / "model.joblib"
metadata_path = ARTIFACT_DIR / "run_metadata.json"

joblib.dump(pipeline, model_path)
metadata_path.write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")

mlflow_status = "not_attempted"
try:
    import mlflow

    mlflow.set_tracking_uri("file:" + str((ARTIFACT_DIR / "mlruns").resolve()))
    mlflow.set_experiment("lesson6_1_mlops_lifecycle")

    with mlflow.start_run(run_name="logreg_baseline"):
        mlflow.log_param("seed", SEED)
        mlflow.log_param("model_type", "LogisticRegression")
        for k, v in metrics.items():
            mlflow.log_metric(k, v)
        mlflow.log_artifact(str(model_path), artifact_path="model")
        mlflow.log_artifact(str(metadata_path), artifact_path="metadata")
    mlflow_status = "logged"
except Exception as exc:
    mlflow_status = f"skipped ({exc})"

{"artifact_dir": str(ARTIFACT_DIR), "mlflow": mlflow_status}

/tmp/ipykernel_576228/4012434090.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_timestamp": datetime.utcnow().isoformat() + "Z",


{'artifact_dir': 'artifacts/lesson6_1',
 'mlflow': "skipped (No module named 'mlflow')"}

## Section D - Lifecycle Mapping (Notebook -> MLOps)

This toy flow maps to the lifecycle as follows:

1. Data snapshotting corresponds to data versioning.
2. Train/evaluate corresponds to experiment iteration.
3. Artifact save + metadata corresponds to packaging and lineage.
4. Optional MLflow tracking corresponds to experiment observability.

In production, these same stages move to automated pipelines with tests, approvals, staged deployments, and monitoring loops.

## Business Case Studies & Exceptions

### Case 1: Team stuck in notebook-only delivery

- **Problem**: models perform well offline but fail to deploy repeatedly.
- **Fix**: define reproducible run metadata, artifact standards, and CI checks before scaling platform complexity.
- **Impact**: faster model handoff and fewer "works on my machine" incidents.

### Case 2: Mature lifecycle in a fintech scoring platform

- **Pattern**: tracked experiments + registry approvals + staged deploy + drift alerts.
- **Impact**: lower release risk and better auditability.

### Exceptions

- For low-impact internal analytics models, full CT may be unnecessary.
- For high-risk models, manual governance gates remain essential even with automation.

## Interview Questions & Answers

1. **Q: What is MLOps in one sentence?**
   **A:** It is the practice of operationalizing ML systems with automation, reproducibility, monitoring, and governance.

2. **Q: Why track experiments?**
   **A:** To reproduce results, compare alternatives, and preserve lineage for audits and debugging.

3. **Q: What is the difference between CI/CD and CT?**
   **A:** CI/CD automates software build/release; CT automates model retraining and validation cycles.

4. **Q: Why save data snapshots?**
   **A:** Metrics are only comparable if evaluated on known, versioned data.

5. **Q: What is model lineage?**
   **A:** End-to-end trace from data + code + parameters to a deployed model version.

6. **Q: When should retraining be triggered?**
   **A:** On schedule or when drift/performance thresholds indicate meaningful degradation.

7. **Q: What is a model registry used for?**
   **A:** Managing model versions, approval states, and deployment metadata.

8. **Q: What is the notebook-to-prod gap?**
   **A:** The gap between exploratory code and production-grade, tested, deployable pipelines.

9. **Q: How do you reduce deployment risk?**
   **A:** Use staged rollout strategies with measurable promotion and rollback criteria.

10. **Q: What is continuous monitoring in ML?**
   **A:** Ongoing tracking of data drift, model behavior, performance, and service health after deployment.

11. **Q: Why are seeds important?**
   **A:** They improve reproducibility and help isolate true changes from random variation.

12. **Q: What is the first MLOps step for a small team?**
   **A:** Standardize experiment metadata and artifact packaging before building full platform automation.
